# Approach 2a2 — Average fit parameters (A, B, n) directly

**Pipeline:** Read `<A>, <B>, <n>` from `Summary/per_run_fit_parameters_bs_{bs}.csv` → evaluate the fit `<A> + <B>/(x+1)^<n>` on the mean_fit BN grid → IPA. **No fitting in this notebook.**

**Caveat:** averaging parameters of a non-linear fit is not formally valid. Included strictly for comparison against 2a1 and 2b.

In [1]:
# === Cell 1 — Config, imports, helpers ===
import os, glob, re
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Paths
BASE_DIR = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\prune_layers_ALL"
FIT_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\Fitting_IPA_curves_data_I"
OUT_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_2a2"
INTERMEDIATE_DIR = os.path.join(OUT_DIR, "intermediate")
os.makedirs(INTERMEDIATE_DIR, exist_ok=True)

BATCH_SIZES = [64, 1024, 60000]
CE_o = np.log(10)   # max CE for 10-class problem, ~2.302585

# Auto-detect pruning percentages from prune_layers_ALL/p-percentage_*/
p_dirs = glob.glob(os.path.join(BASE_DIR, "p-percentage_*"))
PRUNING_LEVELS = sorted([
    float(re.search(r"p-percentage_([\d.]+)", d).group(1))
    for d in p_dirs
])
print(f"Found {len(PRUNING_LEVELS)} pruning percentages: {PRUNING_LEVELS}")
print(f"CE_o = ln(10) = {CE_o:.6f}")

# --- IPA from fit (single source of truth) ---
# CE_L is the physically meaningful learning threshold.
#   CE_L = CE_o - 0.9 * (CE_o - A)
#   IPA  = abs(CE_o - CE_L) / learn_BN   where learn_BN = first BN in x_grid with fitted CE <= CE_L.
# We intentionally do NOT simplify to 0.9*(CE_o - A)/learn_BN — CE_L stays a first-class variable.
def compute_ipa_from_fit(x_grid, A, B, n):
    x_grid = np.asarray(x_grid, dtype=float)
    CE_L = CE_o - 0.9 * (CE_o - A)
    fitted = A + B / ((x_grid + 1) ** n)
    mask = fitted <= CE_L

    if mask.any():
        # In-range: original discrete-search behavior
        learn_BN = float(x_grid[mask][0])
        fitted_at = float(fitted[mask][0])
    else:
        # Out-of-range: extrapolate analytically, then ceil to integer BN
        denom = CE_L - A
        if denom <= 0 or n <= 0 or B <= 0:
            return {"CE_L": CE_L, "learn_BN": np.nan, "IPA": np.nan, "fitted_at_learn_BN": np.nan}
        BN_analytic = (B / denom) ** (1.0 / n) - 1.0
        if not np.isfinite(BN_analytic) or BN_analytic <= 0:
            return {"CE_L": CE_L, "learn_BN": np.nan, "IPA": np.nan, "fitted_at_learn_BN": np.nan}
        learn_BN = float(np.ceil(BN_analytic))
        fitted_at = float(A + B / ((learn_BN + 1) ** n))

    if learn_BN == 0:
        return {"CE_L": CE_L, "learn_BN": 0.0, "IPA": np.nan, "fitted_at_learn_BN": fitted_at}
    IPA = abs(CE_o - CE_L) / learn_BN
    return {"CE_L": CE_L, "learn_BN": learn_BN, "IPA": IPA, "fitted_at_learn_BN": fitted_at}
print("Cell 1 ready.")


Found 19 pruning percentages: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.82, 0.84, 0.86, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 1.0]
CE_o = ln(10) = 2.302585
Cell 1 ready.


In [2]:
# === Cell 2 — Approach 2a2: average the per-run (A, B, n), then IPA ===
# No fitting. For each (P%, BS), read <A>, <B>, <n> from
# Fitting_IPA_curves_data_I/Summary/per_run_fit_parameters_bs_{bs}.csv.
# Borrow the BN grid from mean_fit_p_{p}_bs_{bs}.csv for the discrete search.
# NOTE: averaging parameters of a non-linear fit is not formally justified;
# included only for comparison.
inter_by_bs = {}

for bs in BATCH_SIZES:
    print("\n" + "=" * 70)
    print(f"  Approach 2a2 — Batch size {bs}")
    print("=" * 70)
    params_csv = os.path.join(FIT_DIR, "Summary", f"per_run_fit_parameters_bs_{bs}.csv")
    if not os.path.exists(params_csv):
        print(f"  [SKIP] BS={bs}  — missing {params_csv}")
        continue
    params_df = pd.read_csv(params_csv)
    params_df.columns = params_df.columns.str.strip()

    rows = []
    for p in PRUNING_LEVELS:
        sub = params_df[np.isclose(params_df["Pruning_Percentage"], p * 100)]
        if sub.empty:
            print(f"  [SKIP] P%={p*100:5.1f}%  — no row in {params_csv}")
            continue
        A = float(sub["A_mean"].iloc[0])
        B = float(sub["B_mean"].iloc[0])
        n = float(sub["n_mean"].iloc[0])

        # Borrow BN grid from mean_fit
        grid_csv = os.path.join(FIT_DIR, f"BS_{bs}", f"mean_fit_p_{p}_bs_{bs}.csv")
        if not os.path.exists(grid_csv):
            print(f"  [SKIP] P%={p*100:5.1f}%  — no BN grid file {grid_csv}")
            continue
        grid_df = pd.read_csv(grid_csv)
        grid_df.columns = grid_df.columns.str.strip()
        x_grid = grid_df["Batch_Number"].dropna().values.astype(float)

        ipa = compute_ipa_from_fit(x_grid, A, B, n)

        print(f"  P%={p*100:5.1f}%  <A>={A:.4f}  <B>={B:.4f}  <n>={n:.4f}  "
              f"CE_o={CE_o:.4f}  CE_L={ipa['CE_L']:.4f}  learn_BN={ipa['learn_BN']!r:>8}  IPA={ipa['IPA']}")

        rows.append({
            "P%": p * 100, "A_mean": A, "B_mean": B, "n_mean": n,
            "CE_o": CE_o, "CE_L": ipa["CE_L"],
            "learn_BN": ipa["learn_BN"], "fitted_at_learn_BN": ipa["fitted_at_learn_BN"],
            "IPA": ipa["IPA"],
        })

    if rows:
        bs_df = pd.DataFrame(rows)
        inter_path = os.path.join(INTERMEDIATE_DIR, f"approach_2a2_avg_params_bs_{bs}.csv")
        bs_df.to_csv(inter_path, index=False)
        print(f"  Saved: {inter_path}")
        inter_by_bs[bs] = bs_df

print("\n[Cell 2 done]")



  Approach 2a2 — Batch size 64
  P%=  0.0%  <A>=0.2965  <B>=5.5738  <n>=0.8965  CE_o=2.3026  CE_L=0.4971  learn_BN=    40.0  IPA=0.04513799694437598
  P%= 10.0%  <A>=0.2967  <B>=5.8249  <n>=0.9453  CE_o=2.3026  CE_L=0.4973  learn_BN=    35.0  IPA=0.051580396567576785
  P%= 20.0%  <A>=0.2833  <B>=5.8042  <n>=0.8786  CE_o=2.3026  CE_L=0.4852  learn_BN=    45.0  IPA=0.04038592730326896
  P%= 30.0%  <A>=0.2853  <B>=6.2093  <n>=0.9055  CE_o=2.3026  CE_L=0.4870  learn_BN=    44.0  IPA=0.04126230437075368
  P%= 40.0%  <A>=0.2824  <B>=6.5774  <n>=0.8844  CE_o=2.3026  CE_L=0.4844  learn_BN=    51.0  IPA=0.03565079810224649
  P%= 50.0%  <A>=0.2769  <B>=7.1446  <n>=0.8457  CE_o=2.3026  CE_L=0.4795  learn_BN=    67.0  IPA=0.027210941949032615
  P%= 60.0%  <A>=0.2716  <B>=7.0767  <n>=0.7488  CE_o=2.3026  CE_L=0.4747  learn_BN=   114.0  IPA=0.01603422308099585
  P%= 70.0%  <A>=0.2827  <B>=8.1447  <n>=0.7349  CE_o=2.3026  CE_L=0.4847  learn_BN=   153.0  IPA=0.011881700631680717
  P%= 80.0%  <A>=0.30

  P%= 82.0%  <A>=0.3165  <B>=9.6070  <n>=0.7587  CE_o=2.3026  CE_L=0.5151  learn_BN=   166.0  IPA=0.010768162355551076
  P%= 84.0%  <A>=0.3352  <B>=9.2459  <n>=0.7319  CE_o=2.3026  CE_L=0.5319  learn_BN=   192.0  IPA=0.009222230726846256
  P%= 86.0%  <A>=0.3611  <B>=9.2543  <n>=0.7096  CE_o=2.3026  CE_L=0.5553  learn_BN=   231.0  IPA=0.007564085826118782
  P%= 88.0%  <A>=0.3870  <B>=8.8592  <n>=0.6775  CE_o=2.3026  CE_L=0.5786  learn_BN=   286.0  IPA=0.006028018040822495
  P%= 90.0%  <A>=0.4186  <B>=8.2566  <n>=0.6385  CE_o=2.3026  CE_L=0.6070  learn_BN=   372.0  IPA=0.004558119442760522


  P%= 92.0%  <A>=0.4821  <B>=7.8711  <n>=0.6011  CE_o=2.3026  CE_L=0.6642  learn_BN=   526.0  IPA=0.003114849526651735
  P%= 94.0%  <A>=0.5671  <B>=6.5383  <n>=0.5281  CE_o=2.3026  CE_L=0.7406  learn_BN=   964.0  IPA=0.0016203095004290748
  P%= 96.0%  <A>=0.9443  <B>=3.9588  <n>=0.5007  CE_o=2.3026  CE_L=1.0801  learn_BN=   841.0  IPA=0.0014536123357454427
  P%= 98.0%  <A>=1.5158  <B>=1.4708  <n>=0.5000  CE_o=2.3026  CE_L=1.5945  learn_BN=   349.0  IPA=0.002028988424003175
  P%=100.0%  <A>=2.3000  <B>=0.0000  <n>=0.5000  CE_o=2.3026  CE_L=2.3003  learn_BN=     0.0  IPA=nan
  Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_2a2\intermediate\approach_2a2_avg_params_bs_1024.csv

  Approach 2a2 — Batch size 60000
  P%=  0.0%  <A>=0.2938  <B>=6.7632  <n>=1.0799  CE_o=2.3026  CE_L=0.4947  learn_BN=    25.0  IPA=0.07231446794573076
  P%= 10.0%  <A>=0.2835  <B>=6.3825  <n>=1.0282  CE_o=2.3026  CE_L=0.4854  learn_BN=    28.0  IPA=0.06489951620594203
 

In [3]:
# === Cell 3 — Build wide summary CSV for Approach 2a2 ===
# Schema: P%, IPA_Avg_64, STD_64, IPA_Avg_1024, STD_1024, IPA_Avg_60000, STD_60000
# STD columns blank (NaN) for single-curve approaches; populated only by 2b.
summary_rows = []
for p in PRUNING_LEVELS:
    row = {"P%": p * 100}
    for bs in BATCH_SIZES:
        df = inter_by_bs.get(bs)
        if df is None:
            mean_val, std_val = np.nan, np.nan
        else:
            sub = df[df["P%"] == p * 100]
            mean_val = float(sub["IPA"].iloc[0]) if (not sub.empty and "IPA" in sub.columns) else (
                       float(sub["IPA_mean"].iloc[0]) if (not sub.empty and "IPA_mean" in sub.columns) else np.nan)
        std_val  = np.nan
        row[f"IPA_Avg_{bs}"] = mean_val
        row[f"STD_{bs}"]     = std_val
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows, columns=[
    "P%", "IPA_Avg_64", "STD_64", "IPA_Avg_1024", "STD_1024", "IPA_Avg_60000", "STD_60000"
])
out_csv = os.path.join(OUT_DIR, "ipa_summary_approach_2a2.csv")
summary_df.to_csv(out_csv, index=False)
print(f"\nFinal summary written: {out_csv}")
print(summary_df.to_string(index=False))



Final summary written: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-MNIST\IPA_methods\Approach_2a2\ipa_summary_approach_2a2.csv
   P%  IPA_Avg_64  STD_64  IPA_Avg_1024  STD_1024  IPA_Avg_60000  STD_60000
  0.0    0.045138     NaN      0.065502       NaN       0.072314        NaN
 10.0    0.051580     NaN      0.068046       NaN       0.064900        NaN
 20.0    0.040386     NaN      0.065767       NaN       0.057061        NaN
 30.0    0.041262     NaN      0.055947       NaN       0.050901        NaN
 40.0    0.035651     NaN      0.063470       NaN       0.043750        NaN
 50.0    0.027211     NaN      0.044963       NaN       0.038364        NaN
 60.0    0.016034     NaN      0.048123       NaN       0.029704        NaN
 70.0    0.011882     NaN      0.043233       NaN       0.036491        NaN
 80.0    0.006903     NaN      0.013204       NaN       0.029222        NaN
 82.0    0.005971     NaN      0.010768       NaN       0.016667        NaN
 84.0    0.004974    